# MDC Preprocessing v3 -- T=25 Temporal Window

**Key change from v2:** `WINDOW_SIZE=25`, `STRIDE=5`

| Config | v2 | v3 |
|---|---|---|
| Window size | 10 buckets | **25 buckets** |
| Time covered | 150s (2.5 min) | **375s (6.25 min)** |
| Stride | 2 buckets (30s) | **5 buckets (75s)** |

**Why T=25 improves detection:**
- Port scans, brute-force, and lateral movement take 2-10 min to manifest
- T=10 only captures the tail of slow attacks; T=25 captures the full temporal signature
- STRIDE=5 keeps total window count similar to v2, so RAM usage stays comparable

**All v2 leakage-free guarantees preserved:**
1. Session split happens BEFORE any fitting
2. All transforms (scaler, filters, clip bounds) fit on training data only
3. Outputs `windows_v3.npz` and `preproc_v3.pkl`

## 0. Setup & Config

In [ ]:
import sys, os, subprocess, gc
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'kagglehub[pandas-datasets]', 'joblib'],
        check=False,
    )
    OUTPUT_DIR = os.path.join(os.getcwd(), 'data', 'processed')
else:
    OUTPUT_DIR = os.path.normpath('../data/processed')

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
TEST_RATIO   = 0.15

MIN_FLOWS          = 10_000
GAP_THRESHOLD      = 60
VARIANCE_THRESHOLD = 0.01
CORR_THRESHOLD     = 0.95
BENIGN_LABEL       = 0

# v3 window config
BUCKET_FREQ = '15s'
WINDOW_SIZE = 25    # was 10 -- covers 25x15s = 375s = 6.25 min
STRIDE      = 5     # was 2  -- step 5x15s = 75s between windows

# Fix B: window must be >=50% attack buckets to be labeled attack.
# Prevents dilution: with T=25 and "any attack" rule, a window needs
# only 1/25 (4%) attack concentration, making attack windows look benign.
ATTACK_FRAC_THRESHOLD = 0.5

DROP_COLS = [
    'Flow ID', 'Dst IP', 'Timestamp',
    'Fwd URG Flags', 'Bwd URG Flags',
    'URG Flag Count', 'CWR Flag Count', 'ECE Flag Count',
]

print('v3 config loaded')
print(f'  IN_COLAB              = {_IN_COLAB}')
print(f'  OUTPUT_DIR            = {OUTPUT_DIR}')
print(f'  Window                = {WINDOW_SIZE} x {BUCKET_FREQ} = {WINDOW_SIZE*15}s = {WINDOW_SIZE*15/60:.1f} min')
print(f'  Stride                = {STRIDE} x {BUCKET_FREQ} = {STRIDE*15}s')
print(f'  Attack frac threshold = {ATTACK_FRAC_THRESHOLD} (majority-attack labeling)')

## 1. Load Raw Data

In [ ]:
from pathlib import Path
import kagglehub
from kagglehub import KaggleDatasetAdapter

KAGGLE_DATASET = 'yigitsever/misuse-detection-in-containers-dataset'
KAGGLE_CSV     = 'MDC dataset.csv'

if _IN_COLAB:
    try:
        from google.colab import userdata
        for k, v in [('KAGGLE_USERNAME', userdata.get('KAGGLE_USERNAME')),
                     ('KAGGLE_KEY',      userdata.get('KAGGLE_KEY'))]:
            if v:
                os.environ[k] = v
    except Exception:
        pass

try:
    df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, KAGGLE_DATASET, KAGGLE_CSV)
except Exception:
    root = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    candidates = sorted(root.rglob('*.csv'), key=lambda p: p.stat().st_size, reverse=True)
    if not candidates:
        raise FileNotFoundError(f'No CSV files found under {root}')
    df = pd.read_csv(candidates[0])

df.columns = df.columns.str.strip()
print(f'Raw shape: {df.shape}  |  Labels: {sorted(df["Label"].unique())}')

## 2. Container Filter + Timestamp + Session Assignment

In [ ]:
counts   = df['Src IP'].value_counts()
keep_ips = counts[counts >= MIN_FLOWS].index.tolist()
df       = df[df['Src IP'].isin(keep_ips)].copy().reset_index(drop=True)
print(f'After container filter: {len(df):,} rows, {len(keep_ips)} containers')

df['ts'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df = df.dropna(subset=['ts']).sort_values(['Src IP', 'ts']).reset_index(drop=True)
print(f'After ts parse: {len(df):,} rows')

def assign_sessions(group, threshold=60):
    gap      = group['ts'].diff().dt.total_seconds().fillna(0)
    sess_num = (gap > threshold).cumsum()
    group['session_id'] = group['Src IP'].astype(str) + '_s' + sess_num.astype(str)
    return group

df = df.groupby('Src IP', group_keys=False).apply(assign_sessions, threshold=GAP_THRESHOLD)
df['time_gap_s'] = df.groupby('Src IP')['ts'].diff().dt.total_seconds()

n_sess = df['session_id'].nunique()
print(f'Sessions: {n_sess:,}')
print(df.groupby('Src IP')['session_id'].nunique().to_string())

## 3. Session-Level Random Split + Benign-Only Transform Fitting

Two independent fixes are applied here to prevent score inversion:

**Fix 1 — Random session split (not temporal):**
Temporal split puts early-period benign in training and late-period benign in holdout. Since attacks are temporally concentrated, late benign traffic looks different from early benign → distribution shift → autoencoder scores holdout benign higher than attacks → AUC < 0.5. Random split gives both splits the same temporal distribution.

**Fix 2 — Benign-only transform fitting:**
With random split, training sessions have the same ~65% attack rate as the full dataset. Fitting StandardScaler on 65% attack flows makes attacks look "normal" after normalization → reconstruction error near zero → AUC collapses to 0.10. Fix: filter `df_train` to `Label==0` rows only before any fitting. All transforms (median, variance, correlation, clip bounds, scaler) are then calibrated on the benign distribution. Attack flows in holdout are extreme outliers after this normalization → high reconstruction error → correctly detected.

All downstream transforms fit on training benign data only — no holdout ever seen.

In [ ]:
# ── Low-RAM per-container RANDOM session split + benign-only transform fitting
# WHY random (not temporal):
#   Temporal split → training benign = early sessions, holdout benign = late
#   sessions. Attacks are temporally concentrated, so late-period benign looks
#   "unusual" to the autoencoder → AUC < 0.5 (score inversion).
#   Random split ensures training and holdout benign come from the same distribution.
#
# WHY benign-only filter after split:
#   With random split, training sessions have the same attack rate as the full
#   dataset (~65%). Fitting the StandardScaler on 65% attack flows makes attacks
#   look "normal" after normalization → reconstruction error LOW → AUC collapses
#   to 0.10-0.15 (severe inversion).
#   Fix: retain only Label==0 flows in df_train so every transform (median, var,
#   corr, clip, scaler) is calibrated on the benign distribution only.
#
# Leakage: both guarantees are preserved --
#   (1) scaler/filters fit on training data only (benign subset of training)
#   (2) holdout data never seen during fitting

RANDOM_STATE = 42   # reproducible session shuffle

# Step 1: Early drop of string-heavy columns BEFORE any copy.
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)
gc.collect()

# Step 2: Session assignments from 3-column slice (avoids full-df groupby).
_ss = df[['Src IP', 'session_id', 'ts']].drop_duplicates('session_id').copy()

rng = np.random.default_rng(RANDOM_STATE)
train_sessions   = set()
holdout_sessions = set()
for src_ip, grp in _ss.groupby('Src IP'):
    sess    = grp['session_id'].tolist()
    n_train = max(1, int(len(sess) * TRAIN_RATIO))
    shuffled = rng.permutation(sess)           # random order, fixed seed
    train_sessions.update(shuffled[:n_train])
    holdout_sessions.update(shuffled[n_train:])
del _ss
gc.collect()

# Step 3: Boolean mask
train_mask = df['session_id'].isin(train_sessions)

# Step 4: Extract 4-column metadata BEFORE making full copies (~50 MB total).
KEEP_META    = ['Src IP', 'session_id', 'Label', 'ts']
meta_train   = df.loc[train_mask,  KEEP_META].copy()
meta_holdout = df.loc[~train_mask, KEEP_META].copy()

# Step 5: Stats from meta (no df_train/df_holdout in memory yet).
print(f'Per-container {int(TRAIN_RATIO*100)}/{int((1-TRAIN_RATIO)*100)} RANDOM session split:')
all_ips = sorted(set(meta_train['Src IP'].tolist()) | set(meta_holdout['Src IP'].tolist()))
for src_ip in all_ips:
    tr = meta_train[meta_train['Src IP'] == src_ip]
    ho = meta_holdout[meta_holdout['Src IP'] == src_ip]
    n_tr = len(tr);  n_ho = len(ho)
    att_tr = (tr['Label'] != 0).mean() * 100 if n_tr > 0 else 0.0
    att_ho = (ho['Label'] != 0).mean() * 100 if n_ho > 0 else 0.0
    print(f'  {src_ip}: train={n_tr:,} ({att_tr:.1f}% atk)  holdout={n_ho:,} ({att_ho:.1f}% atk)')
print()
print(f'Train   : {len(meta_train):,} flows  {len(train_sessions):,} sessions')
print(f'  Benign: {(meta_train["Label"]==0).mean()*100:.1f}%  Attack: {(meta_train["Label"]!=0).mean()*100:.1f}%')
print(f'Holdout : {len(meta_holdout):,} flows  {len(holdout_sessions):,} sessions')
print(f'  Benign: {(meta_holdout["Label"]==0).mean()*100:.1f}%  Attack: {(meta_holdout["Label"]!=0).mean()*100:.1f}%')
print(f'Train   time: {meta_train["ts"].min().date()} -> {meta_train["ts"].max().date()}')
print(f'Holdout time: {meta_holdout["ts"].min().date()} -> {meta_holdout["ts"].max().date()}')
print(f'(Overlapping time ranges expected with random split)')

# Step 6: Create df_train and df_holdout.
df_train   = df[train_mask].copy()
df_holdout = df[~train_mask].copy()
del df, train_mask
gc.collect()

# ── Benign-only filter for transform fitting ────────────────────────────────
# All downstream fitting (median, variance, correlation, clip bounds, scaler)
# must see ONLY benign traffic so the normalization is calibrated to normal
# patterns. Attack flows get their extreme values normalized to appear "normal"
# if included here -- making their reconstruction error artificially low.
df_train   = df_train[df_train['Label'] == BENIGN_LABEL].copy()
meta_train = meta_train[meta_train['Label'] == BENIGN_LABEL].copy()
gc.collect()
print(f'\nBenign-only filter applied:')
print(f'  Training flows for fitting: {len(df_train):,}  (all Label==0)')
print(f'  Holdout flows (unchanged) : {len(df_holdout):,}  (benign + attack)')

## 4. Feature Extraction

In [ ]:
drop_all = DROP_COLS + ['ts', 'time_gap_s'] + (['Protocol_Name'] if 'Protocol_Name' in df_train.columns else [])

df_tr = df_train.drop(columns=drop_all + ['Label', 'session_id'], errors='ignore')
df_ho = df_holdout.drop(columns=drop_all + ['Label', 'session_id'], errors='ignore')

df_tr = df_tr.replace([np.inf, -np.inf], np.nan)
df_ho = df_ho.replace([np.inf, -np.inf], np.nan)

df_tr = df_tr.select_dtypes(include=np.number).astype(np.float32)
df_ho = df_ho.select_dtypes(include=np.number).astype(np.float32)

print(f'Train feat  : {df_tr.shape}')
print(f'Holdout feat: {df_ho.shape}')

## 5. NaN Fill -- Train Medians Only

In [ ]:
train_medians = df_tr.median()
df_tr = df_tr.fillna(train_medians)
df_ho = df_ho.fillna(train_medians)
print(f'NaN after fill -- train: {df_tr.isnull().sum().sum()}  holdout: {df_ho.isnull().sum().sum()}')

## 6. Protocol One-Hot (train-derived values)

In [ ]:
proto_tr = df_tr.pop('Protocol').to_numpy(dtype=np.float32)
proto_ho = df_ho.pop('Protocol').to_numpy(dtype=np.float32)
train_proto_vals = sorted([v for v in np.unique(proto_tr) if not np.isnan(v)])

for pv in train_proto_vals:
    name = f'Proto_{int(pv)}'
    df_tr[name] = (proto_tr == pv).astype(np.float32)
    df_ho[name] = (proto_ho == pv).astype(np.float32)

del proto_tr, proto_ho
gc.collect()
print(f'Protocol values from train: {train_proto_vals}')
print(f'Features: {df_tr.shape[1]}')

## 7. Variance Filter -- Fit on Train Only

In [ ]:
feat_names_before_var = df_tr.columns.tolist()
X_var = df_tr.to_numpy(dtype=np.float32, copy=True)

vt = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
vt.fit(X_var)
var_mask = vt.get_support()
del X_var
gc.collect()

keep_var    = [c for c, ok in zip(feat_names_before_var, var_mask) if ok]
dropped_var = [c for c, ok in zip(feat_names_before_var, var_mask) if not ok]

df_tr = df_tr[keep_var]
df_ho = df_ho[keep_var]
print(f'Variance threshold : {VARIANCE_THRESHOLD}')
print(f'Dropped            : {len(dropped_var)} {dropped_var}')
print(f'Remaining          : {df_tr.shape[1]}')

## 8. Correlation Filter -- Fit on Train Sample Only

In [ ]:
n_sample = min(100_000, len(df_tr))
sample   = df_tr.sample(n=n_sample, random_state=42).astype(np.float32, copy=False)

corr      = sample.corr(numeric_only=True).abs()
upper     = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
drop_corr = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]

df_tr = df_tr.drop(columns=drop_corr, errors='ignore')
df_ho = df_ho.drop(columns=drop_corr, errors='ignore')

del sample, corr, upper
gc.collect()
print(f'Correlation threshold : {CORR_THRESHOLD}')
print(f'Dropped               : {len(drop_corr)}')
print(f'Remaining features    : {df_tr.shape[1]}')

## 9. Clip Bounds -- Computed from Train Only

In [ ]:
clip_bounds = {}
n_iqr, n_p99, n_skip = 0, 0, 0

for col in df_tr.columns:
    s = df_tr[col]
    if s.nunique(dropna=False) <= 1:
        clip_bounds[col] = (None, None, 'skip')
        n_skip += 1
        continue
    q1, q3 = float(s.quantile(0.25)), float(s.quantile(0.75))
    iqr = q3 - q1
    if iqr > 0:
        clip_bounds[col] = (q1 - 3 * iqr, q3 + 3 * iqr, 'iqr')
        n_iqr += 1
    else:
        p99 = float(s.quantile(0.99))
        clip_bounds[col] = (None, p99 if p99 > 0 else None, 'p99')
        n_p99 += 1

for col, (lo, hi, strategy) in clip_bounds.items():
    if strategy == 'skip':
        continue
    df_tr[col] = df_tr[col].clip(lower=lo, upper=hi)
    df_ho[col] = df_ho[col].clip(lower=lo, upper=hi)

gc.collect()
print(f'Clipped (+-3xIQR): {n_iqr}  |  (p99 cap): {n_p99}  |  skipped: {n_skip}')

## 10. StandardScaler -- Fit on Train Only

In [ ]:
feat_cols_final = df_tr.columns.tolist()

X_tr_np = df_tr.to_numpy(dtype=np.float32, copy=True)
X_ho_np = df_ho.to_numpy(dtype=np.float32, copy=True)
del df_tr, df_ho
gc.collect()

scaler       = StandardScaler()
X_tr_scaled  = scaler.fit_transform(X_tr_np).astype(np.float32)
X_ho_scaled  = scaler.transform(X_ho_np).astype(np.float32)
del X_tr_np, X_ho_np
gc.collect()

df_tr_proc = pd.DataFrame(X_tr_scaled, columns=feat_cols_final, index=meta_train.index)
df_tr_proc[['Src IP', 'session_id', 'Label', 'ts']] = meta_train[['Src IP', 'session_id', 'Label', 'ts']].values

df_ho_proc = pd.DataFrame(X_ho_scaled, columns=feat_cols_final, index=meta_holdout.index)
df_ho_proc[['Src IP', 'session_id', 'Label', 'ts']] = meta_holdout[['Src IP', 'session_id', 'Label', 'ts']].values

del X_tr_scaled, X_ho_scaled
gc.collect()
print(f'Train processed  : {df_tr_proc.shape}')
print(f'Holdout processed: {df_ho_proc.shape}')
print(f'Feature count    : {len(feat_cols_final)}')

## 11. 15-Second Bucketing (mean + max + std + flow_count)

Aggregates flows into fixed-width time buckets. `flow_count` normalization max is computed from training data only.

In [ ]:
def bucket_flows(df_proc, feature_cols, bucket_freq, benign_label, flow_count_max=None):
    df_proc = df_proc.copy()
    df_proc['ts']     = pd.to_datetime(df_proc['ts'])
    df_proc['bucket'] = df_proc['ts'].dt.floor(bucket_freq)

    g = ['Src IP', 'session_id', 'bucket']

    feat_agg          = df_proc.groupby(g)[feature_cols].agg(['mean', 'max', 'std'])
    feat_agg.columns  = [f'{col}_{stat}' for col, stat in feat_agg.columns]
    feat_agg          = feat_agg.reset_index()

    std_cols = [col for col in feat_agg.columns if col.endswith('_std')]
    feat_agg[std_cols] = feat_agg[std_cols].fillna(0.0)

    cnt = df_proc.groupby(g).size().reset_index(name='_flow_count_raw')

    lbl = df_proc.groupby(g)['Label'].apply(
        lambda x: 0 if (x == benign_label).all() else 1
    ).reset_index(name='label')

    bucket_df = feat_agg.merge(cnt, on=g).merge(lbl, on=g)

    if flow_count_max is None:
        flow_count_max = float(bucket_df['_flow_count_raw'].quantile(0.99))
    bucket_df['flow_count'] = (
        bucket_df['_flow_count_raw'] / flow_count_max
    ).clip(upper=1.0).astype(np.float32)
    bucket_df = bucket_df.drop(columns=['_flow_count_raw'])

    bucket_feature_cols = [col for col in bucket_df.columns if col not in g + ['label']]
    return bucket_df, flow_count_max, bucket_feature_cols

In [ ]:
train_bucket_df, flow_count_max, bucket_feature_cols = bucket_flows(
    df_tr_proc, feat_cols_final, BUCKET_FREQ, BENIGN_LABEL, flow_count_max=None
)
holdout_bucket_df, _, _ = bucket_flows(
    df_ho_proc, feat_cols_final, BUCKET_FREQ, BENIGN_LABEL, flow_count_max=flow_count_max
)

del df_tr_proc, df_ho_proc
gc.collect()

n_tr_b  = len(train_bucket_df);   n_tr_att = train_bucket_df['label'].sum()
n_ho_b  = len(holdout_bucket_df); n_ho_att = holdout_bucket_df['label'].sum()
print(f'Train   buckets: {n_tr_b:,}  attack: {n_tr_att:,} ({n_tr_att/n_tr_b*100:.1f}%)')
print(f'Holdout buckets: {n_ho_b:,}  attack: {n_ho_att:,} ({n_ho_att/n_ho_b*100:.1f}%)')
print(f'Features/bucket: {len(bucket_feature_cols)}')
print(f'flow_count_max (train): {flow_count_max}')

## 12. Zero-Fill Empty Buckets

In [ ]:
def zero_fill_sessions(bucket_df, bucket_freq):
    def fill_one(group):
        src_ip  = group['Src IP'].iloc[0]
        sess_id = group['session_id'].iloc[0]
        full_range = pd.date_range(group['bucket'].min(), group['bucket'].max(), freq=bucket_freq)
        group = (group.set_index('bucket')
                      .drop(columns=['Src IP', 'session_id'])
                      .reindex(full_range, fill_value=0.0))
        group['Src IP']     = src_ip
        group['session_id'] = sess_id
        group['label']      = group['label'].astype(int)
        return group.reset_index().rename(columns={'index': 'bucket'})

    return (bucket_df
            .groupby(['Src IP', 'session_id'], group_keys=False)
            .apply(fill_one)
            .reset_index(drop=True))

train_bucket_df   = zero_fill_sessions(train_bucket_df,   BUCKET_FREQ)
holdout_bucket_df = zero_fill_sessions(holdout_bucket_df, BUCKET_FREQ)
gc.collect()
print(f'Train   buckets after zero-fill: {len(train_bucket_df):,}')
print(f'Holdout buckets after zero-fill: {len(holdout_bucket_df):,}')

## 13. Sliding Windows (T=25, STRIDE=5)

Autoencoder trains on **benign-only** windows.
Val/test contain both benign and attack windows for evaluation.

In [ ]:
def make_windows(bucket_df, W, S, feat_cols, attack_frac_threshold=ATTACK_FRAC_THRESHOLD):
    all_wins, all_lbls = [], []
    for (_, _), grp in bucket_df.groupby(['Src IP', 'session_id']):
        grp  = grp.sort_values('bucket')
        arr  = grp[feat_cols].to_numpy(dtype=np.float32)
        lbls = grp['label'].to_numpy()
        n    = len(arr)
        for start in range(0, n - W + 1, S):
            win_lbls     = lbls[start:start + W]
            attack_frac  = (win_lbls != 0).mean()
            window_label = 1 if attack_frac >= attack_frac_threshold else 0
            all_wins.append(arr[start:start + W])
            all_lbls.append(window_label)
    if not all_wins:
        return np.empty((0, W, len(feat_cols)), dtype=np.float32), np.empty(0, dtype=np.int8)
    return (np.array(all_wins, dtype=np.float32),
            np.array(all_lbls, dtype=np.int8))

X_train_all, y_train_all = make_windows(train_bucket_df,   WINDOW_SIZE, STRIDE, bucket_feature_cols)
X_holdout,   y_holdout   = make_windows(holdout_bucket_df, WINDOW_SIZE, STRIDE, bucket_feature_cols)

del train_bucket_df, holdout_bucket_df
gc.collect()

# Autoencoder trains on BENIGN windows only
benign_mask = (y_train_all == BENIGN_LABEL)
X_train     = X_train_all[benign_mask]
y_train     = y_train_all[benign_mask]
del X_train_all, y_train_all
gc.collect()

n_att = y_holdout.sum()
print(f'Train windows (benign-only)  : {X_train.shape}')
print(f'Holdout windows              : {X_holdout.shape}  attack: {n_att} ({y_holdout.mean()*100:.1f}%)')
print(f'Attack frac threshold applied: {ATTACK_FRAC_THRESHOLD} (majority-attack labeling)')

## 14. Split Holdout -> Val + Test (Stratified 50/50)

In [ ]:
from sklearn.model_selection import train_test_split

X_val, X_test, y_val, y_test = train_test_split(
    X_holdout, y_holdout,
    test_size=0.5,
    stratify=y_holdout,
    random_state=42,
)

print(f'X_train : {X_train.shape}  (benign only)')
print(f'X_val   : {X_val.shape}   attack: {y_val.mean()*100:.1f}%')
print(f'X_test  : {X_test.shape}  attack: {y_test.mean()*100:.1f}%')

## 15. Save Artifacts

In [ ]:
import joblib
from datetime import datetime, timezone

npz_path = f'{OUTPUT_DIR}/windows_v3.npz'
np.savez_compressed(
    npz_path,
    X_train=X_train, X_val=X_val, X_test=X_test,
    y_val=y_val, y_test=y_test,
)
print(f'Saved windows  : {npz_path}  ({os.path.getsize(npz_path)/1024/1024:.1f} MB)')

preproc_v3 = {
    'version'              : 'v3',
    'created_at'           : datetime.now(timezone.utc).isoformat(),
    'window_size'          : WINDOW_SIZE,
    'stride'               : STRIDE,
    'bucket_freq'          : BUCKET_FREQ,
    'benign_label'         : BENIGN_LABEL,
    'train_medians'        : train_medians,
    'train_proto_vals'     : train_proto_vals,
    'feat_names_before_var': feat_names_before_var,
    'keep_var_mask'        : var_mask,
    'drop_corr'            : drop_corr,
    'clip_bounds'          : clip_bounds,
    'scaler'               : scaler,
    'feat_cols_final'      : feat_cols_final,
    'bucket_feature_cols'  : bucket_feature_cols,
    'flow_count_max'       : flow_count_max,
}
pkl_path = f'{OUTPUT_DIR}/preproc_v3.pkl'
joblib.dump(preproc_v3, pkl_path)
print(f'Saved preproc  : {pkl_path}')
print(f'Shapes: X_train={X_train.shape}  X_val={X_val.shape}  X_test={X_test.shape}')

## 16. Leakage Validation

In [ ]:
print('=' * 60)
print('LEAKAGE VALIDATION REPORT  (v3)')
print('=' * 60)

# [1] No session in both splits
overlap = train_sessions & holdout_sessions
print(f'[1] Session overlap: {len(overlap)}  -> {"PASS" if len(overlap)==0 else "FAIL"}')

# [2] Scaler fitted on train only
print(f'[2] Scaler fitted on train only ({len(meta_train):,} flows)')
print(f'    scaler.mean_[:5] = {scaler.mean_[:5].round(6)}  -> PASS')

# [3] Per-container coverage -- every container in both train and holdout.
print('[3] Per-container coverage (all containers in both train & holdout):')
all_ok = True
for src_ip in sorted(meta_train['Src IP'].unique()):
    in_tr = src_ip in meta_train['Src IP'].values
    in_ho = src_ip in meta_holdout['Src IP'].values
    ok    = in_tr and in_ho
    if not ok:
        all_ok = False
    print(f'    {src_ip}: train={in_tr}  holdout={in_ho}  -> {"PASS" if ok else "WARN"}')
print(f'    Overall -> {"PASS" if all_ok else "WARN: some container missing from one split"}')

# [3b] Time range overlap -- with random split BOTH splits span the full range.
# This is the key health check: overlapping ranges confirm no temporal bias.
tr_min = meta_train['ts'].min().date();    tr_max = meta_train['ts'].max().date()
ho_min = meta_holdout['ts'].min().date();  ho_max = meta_holdout['ts'].max().date()
overlap_ok = (tr_min == ho_min) and (tr_max == ho_max)
print(f'[3b] Time ranges (random split -- should OVERLAP):')
print(f'    Train  : {tr_min} -> {tr_max}')
print(f'    Holdout: {ho_min} -> {ho_max}')
print(f'    -> {"PASS (same range, no temporal bias)" if overlap_ok else "WARN (ranges differ -- check split)"}')

# [4] Clip bounds from training only
n_iqr_b = sum(1 for _, _, s in clip_bounds.values() if s == 'iqr')
print(f'[4] Clip bounds: {n_iqr_b} IQR-based -- all from training data  -> PASS')

# [5] Training windows are benign-only
print(f'[5] X_train benign-only: {(y_train == 0).all()}  -> PASS')

# [6] Window label sanity: attack rate and majority-attack threshold
att_frac = y_holdout.mean()
print(f'[6] Holdout attack rate: {att_frac*100:.1f}%  (ATTACK_FRAC_THRESHOLD={ATTACK_FRAC_THRESHOLD})')
print(f'    Attack windows need >=50% attack buckets  -> label dilution fix ACTIVE')
print('=' * 60)

## 17. Export to Google Drive

In [ ]:
import shutil, json

DRIVE_SUBDIR = 'Module4_MDC/processed_v3'
ARTIFACTS    = ['windows_v3.npz', 'preproc_v3.pkl']

if _IN_COLAB:
    from google.colab import drive
    _dr = '/content/drive'
    if not os.path.isdir(os.path.join(_dr, 'MyDrive')):
        os.makedirs(_dr, exist_ok=True)
        drive.mount(_dr)
    dst = Path('/content/drive/MyDrive') / DRIVE_SUBDIR
    dst.mkdir(parents=True, exist_ok=True)
    for name in ARTIFACTS:
        src = Path(OUTPUT_DIR) / name
        if src.is_file():
            shutil.copy2(src, dst / name)
            print(f'Copied -> {dst / name}')
    manifest = {
        'exported_at'        : datetime.now(timezone.utc).isoformat(),
        'version'            : 'v3',
        'shapes'             : {
            'X_train': list(X_train.shape),
            'X_val'  : list(X_val.shape),
            'X_test' : list(X_test.shape),
        },
        'bucket_freq'        : BUCKET_FREQ,
        'window_size'        : WINDOW_SIZE,
        'stride'             : STRIDE,
        'features_per_bucket': len(bucket_feature_cols),
        'leakage_free'       : True,
    }
    (dst / 'manifest_v3.json').write_text(json.dumps(manifest, indent=2))
    print(f'Drive export complete -> {dst}')
else:
    print(f'Not on Colab -- files saved locally to {OUTPUT_DIR}')